## Extracción del Archivo Tratado Eliminación de Columnas Irrelevantes

In [ ]:
import pandas as pd

df = pd.read_csv('datos_tratados.csv')

In [ ]:
df.head()

In [ ]:
col = df.columns

In [ ]:
df.info()

In [ ]:
columnas_a_borrar = ['genero','cargos_totales','diarias']

df = df.drop(columns=columnas_a_borrar)
df.columns

In [ ]:
df.info()

## Encoding

In [ ]:
columnas_cats = df.select_dtypes(include=['object', 'category']).columns

for col in columnas_cats:
    print(f'{col.upper()}')
    print(df[col].unique())

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, drop='first')

encoded_data = encoder.fit_transform(df[columnas_cats])

encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out())

df = pd.concat([df, encoded_df], axis=1).drop(columns=columnas_cats)

df

## Verificación de la Proporción de Cancelación (Churn)

In [ ]:
print(df['evasion'].value_counts())
print((df['evasion'].value_counts(normalize=True)*100).round(2))

## Balanceo de Clases

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [ ]:
x = df.drop(columns=['evasion'])
y = df['evasion']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)

x_train_res, y_train_res = smote.fit_resample(x_train, y_train)

print('Antes de SMOTE:', y_train.value_counts())
print('Despues de SMOTE:', y_train_res.value_counts())

## Normalización o Estandarización

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train_res)

x_test_scaled = scaler.transform(x_test)

## Análisis de Correlación

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_numerico = df.select_dtypes(include=['number'])

matriz_corr = df_numerico.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt='.2f',linewidths=0.5, center=0)

plt.title('Matriz de Correlacion: Factores que influyen en la evasion', fontsize=15)
plt.show()

# 4. Extraemos específicamente la relación con la variable 'evasion'
correlacion_objetivo = matriz_corr['evasion'].sort_values(ascending=False)
print("--- Correlación de cada variable con la Evasión ---")
print(correlacion_objetivo)